<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/716_%ED%85%90%EC%84%9C%ED%94%8C%EB%A1%9C%EC%9A%B0_%EC%9E%90%EC%9C%A8%EC%A3%BC%ED%96%89_%EC%B0%A8%EC%84%A0(%EC%95%BC%EA%B0%84).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🌟 주요 특징
1. 밤 영상 특화 전처리

CLAHE: 어두운 영상의 대비 향상
감마 보정: 밝기 조절로 차선 가시성 향상
LAB 색공간: 조명 변화에 강한 처리

2. 딥러닝 모델

U-Net: 세그멘테이션에 최적화된 아키텍처
Simple CNN: 빠른 테스트용 경량 모델
Dice Loss: 세그멘테이션 성능 향상

3. 합성 데이터 생성

실제 데이터가 없을 때 테스트용
밤 도로 환경 시뮬레이션

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os
from pathlib import Path
import json
from IPython.display import display, HTML
import ipywidgets as widgets
from io import BytesIO
import base64
from PIL import Image

# GPU 메모리 증가 설정
physical_devices = tf.config.experimental.list_physical_devices('GPU')
if physical_devices:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

class NightLaneDetector:
    def __init__(self, input_height=160, input_width=320):
        self.input_height = input_height
        self.input_width = input_width
        self.model = None

    def preprocess_image(self, image_data):
        """이미지 전처리 - 밤 도로에 특화"""
        # 이미지가 파일 경로인지 numpy 배열인지 확인
        if isinstance(image_data, str):
            # 파일 경로인 경우
            image = cv2.imread(image_data)
            if image is None:
                raise ValueError(f"이미지를 불러올 수 없습니다: {image_data}")
        else:
            # numpy 배열인 경우 (업로드된 이미지)
            image = image_data

        # BGR to RGB
        if len(image.shape) == 3 and image.shape[2] == 3:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 리사이즈
        image = cv2.resize(image, (self.input_width, self.input_height))

        # 밤 영상 개선
        image = self.enhance_night_image(image)

        # 정규화
        image = image.astype(np.float32) / 255.0

        return image

    def enhance_night_image(self, image):
        """밤 영상 화질 개선"""
        # LAB 색공간 변환
        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)

        # CLAHE (Contrast Limited Adaptive Histogram Equalization) 적용
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        l = clahe.apply(l)

        # LAB 채널 합치기
        enhanced = cv2.merge([l, a, b])

        # RGB로 변환
        enhanced = cv2.cvtColor(enhanced, cv2.COLOR_LAB2RGB)

        # 감마 보정 (밝기 조정)
        gamma = 1.2
        enhanced = np.power(enhanced / 255.0, gamma) * 255.0
        enhanced = enhanced.astype(np.uint8)

        return enhanced

    def create_simple_model(self):
        """간단한 CNN 모델 (빠른 테스트용)"""
        inputs = tf.keras.Input(shape=(self.input_height, self.input_width, 3))

        # 인코더
        x = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
        x = tf.keras.layers.MaxPooling2D(2, 2)(x)

        x = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x)
        x = tf.keras.layers.MaxPooling2D(2, 2)(x)

        x = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(x)
        x = tf.keras.layers.MaxPooling2D(2, 2)(x)

        # 디코더
        x = tf.keras.layers.UpSampling2D(2)(x)
        x = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x)

        x = tf.keras.layers.UpSampling2D(2)(x)
        x = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(x)

        x = tf.keras.layers.UpSampling2D(2)(x)
        x = tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same')(x)

        # 출력 레이어 - 원본 크기와 동일하게 출력
        outputs = tf.keras.layers.Conv2D(1, 1, activation='sigmoid', padding='same')(x)

        model = tf.keras.Model(inputs=inputs, outputs=outputs)
        return model

    def dice_coefficient(self, y_true, y_pred, smooth=1):
        """Dice 계수 - 세그멘테이션 평가 지표"""
        y_true_f = tf.keras.backend.flatten(y_true)
        y_pred_f = tf.keras.backend.flatten(y_pred)
        intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
        return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

    def dice_loss(self, y_true, y_pred):
        """Dice 손실 함수"""
        return 1 - self.dice_coefficient(y_true, y_pred)

    def compile_model(self):
        """모델 컴파일"""
        self.model = self.create_simple_model()

        self.model.compile(
            optimizer='adam',
            loss=self.dice_loss,
            metrics=[self.dice_coefficient, 'accuracy']
        )

        return self.model

    def create_synthetic_data(self, num_samples=1000):
        """합성 데이터 생성 (실제 데이터 없을 때 테스트용)"""
        print("합성 데이터 생성 중...")

        X = []
        y = []

        for i in range(num_samples):
            # 어두운 배경 생성
            img = np.random.randint(0, 30, (self.input_height, self.input_width, 3), dtype=np.uint8)

            # 차선 추가
            mask = np.zeros((self.input_height, self.input_width, 1), dtype=np.uint8)

            # 왼쪽 차선
            left_lane_x = np.random.randint(50, 100)
            cv2.line(img, (left_lane_x, self.input_height-1), (left_lane_x + 30, 0), (255, 255, 255), 3)
            cv2.line(mask[:,:,0], (left_lane_x, self.input_height-1), (left_lane_x + 30, 0), 255, 3)

            # 오른쪽 차선
            right_lane_x = np.random.randint(220, 270)
            cv2.line(img, (right_lane_x, self.input_height-1), (right_lane_x - 30, 0), (255, 255, 255), 3)
            cv2.line(mask[:,:,0], (right_lane_x, self.input_height-1), (right_lane_x - 30, 0), 255, 3)

            # 노이즈 추가
            noise = np.random.randint(0, 50, img.shape, dtype=np.uint8)
            img = cv2.add(img, noise)

            # 정규화
            img = img.astype(np.float32) / 255.0
            mask = mask.astype(np.float32) / 255.0

            X.append(img)
            y.append(mask)

        return np.array(X), np.array(y)

    def train_model(self, X_train, y_train, epochs=20, batch_size=16):
        """모델 훈련"""
        print(f"모델 훈련 시작 - 에폭: {epochs}, 배치 크기: {batch_size}")

        # 콜백 설정
        callbacks = [
            tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
        ]

        # 훈련
        history = self.model.fit(
            X_train, y_train,
            batch_size=batch_size,
            epochs=epochs,
            validation_split=0.2,
            callbacks=callbacks,
            verbose=1
        )

        return history

    def predict_lanes(self, image_data):
        """차선 예측"""
        if self.model is None:
            raise ValueError("모델이 훈련되지 않았습니다.")

        # 이미지 전처리
        processed_image = self.preprocess_image(image_data)

        # 예측
        prediction = self.model.predict(np.expand_dims(processed_image, axis=0))
        lane_mask = prediction[0, :, :, 0]

        return processed_image, lane_mask

    def visualize_result(self, original_image, lane_mask, threshold=0.5):
        """결과 시각화"""
        # 이진화
        binary_mask = (lane_mask > threshold).astype(np.uint8) * 255

        # 원본 이미지 복사
        result = original_image.copy()

        # 차선 영역을 빨간색으로 표시
        lane_pixels = binary_mask > 0
        result[lane_pixels] = [1.0, 0.0, 0.0]  # 빨간색

        # 결과 출력
        plt.figure(figsize=(15, 5))

        plt.subplot(1, 3, 1)
        plt.imshow(original_image)
        plt.title('Preprocessed original image')
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.imshow(lane_mask, cmap='gray')
        plt.title('차선 예측 마스크')
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.imshow(result)
        plt.title('Suboptimal prediction mask')
        plt.axis('off')

        plt.tight_layout()
        plt.show()

    def save_model(self, filepath):
        """모델 저장"""
        self.model.save(filepath)
        print(f"모델이 저장되었습니다: {filepath}")

    def load_model(self, filepath):
        """모델 로드"""
        self.model = tf.keras.models.load_model(
            filepath,
            custom_objects={
                'dice_coefficient': self.dice_coefficient,
                'dice_loss': self.dice_loss
            }
        )
        print(f"모델이 로드되었습니다: {filepath}")


class FileUploadInterface:
    """파일 업로드 인터페이스"""

    def __init__(self, detector):
        self.detector = detector
        self.uploaded_image = None
        self.create_interface()

    def create_interface(self):
        """업로드 인터페이스 생성"""
        # 파일 업로드 위젯
        self.upload_widget = widgets.FileUpload(
            accept='image/*',  # 이미지 파일만 허용
            multiple=False,    # 단일 파일만
            description='영상 업로드'
        )

        # 분석 버튼
        self.analyze_button = widgets.Button(
            description='차선 분석 시작',
            button_style='success',
            disabled=True,
            layout=widgets.Layout(width='200px', height='40px')
        )

        # 결과 출력 영역
        self.output = widgets.Output()

        # 이벤트 핸들러 연결
        self.upload_widget.observe(self.on_upload, names='value')
        self.analyze_button.on_click(self.on_analyze)

        # 레이아웃 구성
        self.interface = widgets.VBox([
            widgets.HTML("<h2>🚗 밤 도로 차선 인식 시스템</h2>"),
            widgets.HTML("<p>밤 도로 영상을 업로드하면 AI가 차선을 검출합니다.</p>"),
            self.upload_widget,
            self.analyze_button,
            self.output
        ])

    def on_upload(self, change):
        """파일 업로드 시 호출"""
        if change['new']:
            uploaded_file = list(change['new'].values())[0]

            try:
                # 이미지 데이터 읽기
                image_data = uploaded_file['content']

                # PIL로 이미지 열기
                image = Image.open(BytesIO(image_data))

                # numpy 배열로 변환
                self.uploaded_image = np.array(image)

                # 분석 버튼 활성화
                self.analyze_button.disabled = False

                with self.output:
                    self.output.clear_output()
                    print(f"✅ 이미지 업로드 완료: {uploaded_file['metadata']['name']}")
                    print(f"이미지 크기: {self.uploaded_image.shape}")

                    # 업로드된 이미지 미리보기
                    plt.figure(figsize=(10, 6))
                    plt.imshow(self.uploaded_image)
                    plt.title('업로드된 이미지')
                    plt.axis('off')
                    plt.show()

            except Exception as e:
                with self.output:
                    self.output.clear_output()
                    print(f"❌ 이미지 로드 실패: {e}")

    def on_analyze(self, button):
        """차선 분석 버튼 클릭 시 호출"""
        if self.uploaded_image is None:
            with self.output:
                print("❌ 먼저 이미지를 업로드해주세요.")
            return

        with self.output:
            self.output.clear_output()
            print("🔍 차선 분석 중...")

            try:
                # 차선 예측
                original_img, lane_mask = self.detector.predict_lanes(self.uploaded_image)

                # 결과 시각화
                self.detector.visualize_result(original_img, lane_mask)

                # 성능 정보 출력
                confidence = np.mean(lane_mask)
                print(f"\n📊 분석 결과:")
                print(f"차선 검출 신뢰도: {confidence:.3f}")

                if confidence > 0.1:
                    print("✅ 차선이 검출되었습니다!")
                else:
                    print("⚠️  차선 검출이 어렵습니다.")
                    print("💡 팁: 더 선명한 차선이 있는 이미지를 사용해보세요.")

            except Exception as e:
                print(f"❌ 분석 중 오류 발생: {e}")

    def display(self):
        """인터페이스 표시"""
        return self.interface


def initialize_system():
    """시스템 초기화"""
    print("=== 밤 도로 차선 인식 시스템 초기화 ===")

    # 모델 초기화
    detector = NightLaneDetector()

    # 모델 생성 및 컴파일
    print("\n1. 모델 생성 중...")
    model = detector.compile_model()

    # 합성 데이터로 기본 훈련
    print("\n2. 기본 훈련 중 (합성 데이터)...")
    X_train, y_train = detector.create_synthetic_data(num_samples=200)  # 빠른 훈련을 위해 적은 샘플

    # 빠른 훈련
    history = detector.train_model(X_train, y_train, epochs=10, batch_size=8)

    print("\n3. 시스템 준비 완료! ✅")

    return detector


def main():
    """메인 실행 함수"""
    # 시스템 초기화
    detector = initialize_system()

    # 파일 업로드 인터페이스 생성
    upload_interface = FileUploadInterface(detector)

    print("\n🎯 사용 방법:")
    print("1. 아래 '영상 업로드' 버튼을 클릭")
    print("2. 밤 도로 이미지 파일 선택 (jpg, png 등)")
    print("3. '차선 분석 시작' 버튼 클릭")
    print("4. 결과 확인!")

    # 인터페이스 표시
    return upload_interface.display()


# Colab에서 바로 실행할 수 있는 함수
def quick_test_with_file(image_path, detector=None):
    """파일 경로로 빠른 테스트"""
    if detector is None:
        detector = initialize_system()

    try:
        print(f"🔍 이미지 분석 중: {image_path}")

        # 차선 예측
        original_img, lane_mask = detector.predict_lanes(image_path)

        # 결과 시각화
        detector.visualize_result(original_img, lane_mask)

        # 성능 정보
        confidence = np.mean(lane_mask)
        print(f"\n📊 차선 검출 신뢰도: {confidence:.3f}")

        if confidence > 0.1:
            print("✅ 차선이 검출되었습니다!")
        else:
            print("⚠️  차선 검출이 어렵습니다.")

    except Exception as e:
        print(f"❌ 에러: {e}")


if __name__ == "__main__":
    # 웹 인터페이스 실행
    interface = main()
    display(interface)

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os
from pathlib import Path
import json
from IPython.display import display, HTML
import ipywidgets as widgets
from io import BytesIO
import base64
from PIL import Image

# GPU 메모리 증가 설정
physical_devices = tf.config.experimental.list_physical_devices('GPU')
if physical_devices:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

class NightLaneDetector:
    def __init__(self, input_height=160, input_width=320):
        self.input_height = input_height
        self.input_width = input_width
        self.model = None

    def preprocess_image(self, image_data):
        """이미지 전처리 - 밤 도로에 특화"""
        # 이미지가 파일 경로인지 numpy 배열인지 확인
        if isinstance(image_data, str):
            # 파일 경로인 경우
            image = cv2.imread(image_data)
            if image is None:
                raise ValueError(f"이미지를 불러올 수 없습니다: {image_data}")
        else:
            # numpy 배열인 경우 (업로드된 이미지)
            image = image_data

        # BGR to RGB
        if len(image.shape) == 3 and image.shape[2] == 3:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 리사이즈
        image = cv2.resize(image, (self.input_width, self.input_height))

        # 밤 영상 개선
        image = self.enhance_night_image(image)

        # 정규화
        image = image.astype(np.float32) / 255.0

        return image

    def enhance_night_image(self, image):
        """밤 영상 화질 개선"""
        # LAB 색공간 변환
        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)

        # CLAHE (Contrast Limited Adaptive Histogram Equalization) 적용
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        l = clahe.apply(l)

        # LAB 채널 합치기
        enhanced = cv2.merge([l, a, b])

        # RGB로 변환
        enhanced = cv2.cvtColor(enhanced, cv2.COLOR_LAB2RGB)

        # 감마 보정 (밝기 조정)
        gamma = 1.2
        enhanced = np.power(enhanced / 255.0, gamma) * 255.0
        enhanced = enhanced.astype(np.uint8)

        return enhanced

    def create_simple_model(self):
        """간단한 CNN 모델 (빠른 테스트용)"""
        inputs = tf.keras.Input(shape=(self.input_height, self.input_width, 3))

        # 인코더
        x = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
        x = tf.keras.layers.MaxPooling2D(2, 2)(x)

        x = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x)
        x = tf.keras.layers.MaxPooling2D(2, 2)(x)

        x = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(x)
        x = tf.keras.layers.MaxPooling2D(2, 2)(x)

        # 디코더
        x = tf.keras.layers.UpSampling2D(2)(x)
        x = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x)

        x = tf.keras.layers.UpSampling2D(2)(x)
        x = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(x)

        x = tf.keras.layers.UpSampling2D(2)(x)
        x = tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same')(x)

        # 출력 레이어 - 원본 크기와 동일하게 출력
        outputs = tf.keras.layers.Conv2D(1, 1, activation='sigmoid', padding='same')(x)

        model = tf.keras.Model(inputs=inputs, outputs=outputs)
        return model

    def dice_coefficient(self, y_true, y_pred, smooth=1):
        """Dice 계수 - 세그멘테이션 평가 지표"""
        y_true_f = tf.keras.backend.flatten(y_true)
        y_pred_f = tf.keras.backend.flatten(y_pred)
        intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
        return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

    def dice_loss(self, y_true, y_pred):
        """Dice 손실 함수"""
        return 1 - self.dice_coefficient(y_true, y_pred)

    def compile_model(self):
        """모델 컴파일"""
        self.model = self.create_simple_model()

        self.model.compile(
            optimizer='adam',
            loss=self.dice_loss,
            metrics=[self.dice_coefficient, 'accuracy']
        )

        return self.model

    def create_synthetic_data(self, num_samples=1000):
        """합성 데이터 생성 (실제 데이터 없을 때 테스트용)"""
        print("합성 데이터 생성 중...")

        X = []
        y = []

        for i in range(num_samples):
            # 어두운 배경 생성
            img = np.random.randint(0, 30, (self.input_height, self.input_width, 3), dtype=np.uint8)

            # 차선 추가
            mask = np.zeros((self.input_height, self.input_width, 1), dtype=np.uint8)

            # 왼쪽 차선
            left_lane_x = np.random.randint(50, 100)
            cv2.line(img, (left_lane_x, self.input_height-1), (left_lane_x + 30, 0), (255, 255, 255), 3)
            cv2.line(mask[:,:,0], (left_lane_x, self.input_height-1), (left_lane_x + 30, 0), 255, 3)

            # 오른쪽 차선
            right_lane_x = np.random.randint(220, 270)
            cv2.line(img, (right_lane_x, self.input_height-1), (right_lane_x - 30, 0), (255, 255, 255), 3)
            cv2.line(mask[:,:,0], (right_lane_x, self.input_height-1), (right_lane_x - 30, 0), 255, 3)

            # 노이즈 추가
            noise = np.random.randint(0, 50, img.shape, dtype=np.uint8)
            img = cv2.add(img, noise)

            # 정규화
            img = img.astype(np.float32) / 255.0
            mask = mask.astype(np.float32) / 255.0

            X.append(img)
            y.append(mask)

        return np.array(X), np.array(y)

    def train_model(self, X_train, y_train, epochs=20, batch_size=16):
        """모델 훈련"""
        print(f"모델 훈련 시작 - 에폭: {epochs}, 배치 크기: {batch_size}")

        # 콜백 설정
        callbacks = [
            tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
        ]

        # 훈련
        history = self.model.fit(
            X_train, y_train,
            batch_size=batch_size,
            epochs=epochs,
            validation_split=0.2,
            callbacks=callbacks,
            verbose=1
        )

        return history

    def predict_lanes(self, image_data):
        """차선 예측"""
        if self.model is None:
            raise ValueError("모델이 훈련되지 않았습니다.")

        # 이미지 전처리
        processed_image = self.preprocess_image(image_data)

        # 예측
        prediction = self.model.predict(np.expand_dims(processed_image, axis=0))
        lane_mask = prediction[0, :, :, 0]

        return processed_image, lane_mask

    def visualize_result(self, original_image, lane_mask, threshold=0.5):
        """결과 시각화"""
        # 이진화
        binary_mask = (lane_mask > threshold).astype(np.uint8) * 255

        # 원본 이미지 복사
        result = original_image.copy()

        # 차선 영역을 빨간색으로 표시
        lane_pixels = binary_mask > 0
        result[lane_pixels] = [1.0, 0.0, 0.0]  # 빨간색

        # 결과 출력
        plt.figure(figsize=(15, 5))

        plt.subplot(1, 3, 1)
        plt.imshow(original_image)
        plt.title('전처리된 원본 이미지')
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.imshow(lane_mask, cmap='gray')
        plt.title('차선 예측 마스크')
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.imshow(result)
        plt.title('차선 검출 결과')
        plt.axis('off')

        plt.tight_layout()
        plt.show()

    def save_model(self, filepath):
        """모델 저장"""
        self.model.save(filepath)
        print(f"모델이 저장되었습니다: {filepath}")

    def load_model(self, filepath):
        """모델 로드"""
        self.model = tf.keras.models.load_model(
            filepath,
            custom_objects={
                'dice_coefficient': self.dice_coefficient,
                'dice_loss': self.dice_loss
            }
        )
        print(f"모델이 로드되었습니다: {filepath}")


class FileUploadInterface:
    """파일 업로드 인터페이스"""

    def __init__(self, detector):
        self.detector = detector
        self.uploaded_image = None
        self.create_interface()

    def create_interface(self):
        """업로드 인터페이스 생성"""
        # 파일 업로드 위젯
        self.upload_widget = widgets.FileUpload(
            accept='image/*',  # 이미지 파일만 허용
            multiple=False,    # 단일 파일만
            description='영상 업로드'
        )

        # 분석 버튼
        self.analyze_button = widgets.Button(
            description='차선 분석 시작',
            button_style='success',
            disabled=True,
            layout=widgets.Layout(width='200px', height='40px')
        )

        # 결과 출력 영역
        self.output = widgets.Output()

        # 이벤트 핸들러 연결
        self.upload_widget.observe(self.on_upload, names='value')
        self.analyze_button.on_click(self.on_analyze)

        # 레이아웃 구성
        self.interface = widgets.VBox([
            widgets.HTML("<h2>🚗 밤 도로 차선 인식 시스템</h2>"),
            widgets.HTML("<p>밤 도로 영상을 업로드하면 AI가 차선을 검출합니다.</p>"),
            self.upload_widget,
            self.analyze_button,
            self.output
        ])

    def on_upload(self, change):
        """파일 업로드 시 호출"""
        if change['new']:
            uploaded_file = list(change['new'].values())[0]

            try:
                # 이미지 데이터 읽기
                image_data = uploaded_file['content']

                # PIL로 이미지 열기
                image = Image.open(BytesIO(image_data))

                # numpy 배열로 변환
                self.uploaded_image = np.array(image)

                # 분석 버튼 활성화
                self.analyze_button.disabled = False

                with self.output:
                    self.output.clear_output()
                    print(f"✅ 이미지 업로드 완료: {uploaded_file['metadata']['name']}")
                    print(f"이미지 크기: {self.uploaded_image.shape}")

                    # 업로드된 이미지 미리보기
                    plt.figure(figsize=(10, 6))
                    plt.imshow(self.uploaded_image)
                    plt.title('업로드된 이미지')
                    plt.axis('off')
                    plt.show()

            except Exception as e:
                with self.output:
                    self.output.clear_output()
                    print(f"❌ 이미지 로드 실패: {e}")

    def on_analyze(self, button):
        """차선 분석 버튼 클릭 시 호출"""
        if self.uploaded_image is None:
            with self.output:
                print("❌ 먼저 이미지를 업로드해주세요.")
            return

        with self.output:
            self.output.clear_output()
            print("🔍 차선 분석 중...")

            try:
                # 차선 예측
                original_img, lane_mask = self.detector.predict_lanes(self.uploaded_image)

                # 결과 시각화
                self.detector.visualize_result(original_img, lane_mask)

                # 성능 정보 출력
                confidence = np.mean(lane_mask)
                print(f"\n📊 분석 결과:")
                print(f"차선 검출 신뢰도: {confidence:.3f}")

                if confidence > 0.1:
                    print("✅ 차선이 검출되었습니다!")
                else:
                    print("⚠️  차선 검출이 어렵습니다.")
                    print("💡 팁: 더 선명한 차선이 있는 이미지를 사용해보세요.")

            except Exception as e:
                print(f"❌ 분석 중 오류 발생: {e}")

    def display(self):
        """인터페이스 표시"""
        return self.interface


def initialize_system():
    """시스템 초기화"""
    print("=== 밤 도로 차선 인식 시스템 초기화 ===")

    # 모델 초기화
    detector = NightLaneDetector()

    # 모델 생성 및 컴파일
    print("\n1. 모델 생성 중...")
    model = detector.compile_model()

    # 합성 데이터로 기본 훈련
    print("\n2. 기본 훈련 중 (합성 데이터)...")
    X_train, y_train = detector.create_synthetic_data(num_samples=200)  # 빠른 훈련을 위해 적은 샘플

    # 빠른 훈련
    history = detector.train_model(X_train, y_train, epochs=10, batch_size=8)

    print("\n3. 시스템 준비 완료! ✅")

    return detector


def main():
    """메인 실행 함수"""
    # 시스템 초기화
    detector = initialize_system()

    # 파일 업로드 인터페이스 생성
    upload_interface = FileUploadInterface(detector)

    print("\n🎯 사용 방법:")
    print("1. 아래 '영상 업로드' 버튼을 클릭")
    print("2. 밤 도로 이미지 파일 선택 (jpg, png 등)")
    print("3. '차선 분석 시작' 버튼 클릭")
    print("4. 결과 확인!")

    # 인터페이스 표시
    return upload_interface.display()


# Colab에서 바로 실행할 수 있는 함수
def quick_test_with_file(image_path, detector=None):
    """파일 경로로 빠른 테스트"""
    if detector is None:
        detector = initialize_system()

    try:
        print(f"🔍 이미지 분석 중: {image_path}")

        # 차선 예측
        original_img, lane_mask = detector.predict_lanes(image_path)

        # 결과 시각화
        detector.visualize_result(original_img, lane_mask)

        # 성능 정보
        confidence = np.mean(lane_mask)
        print(f"\n📊 차선 검출 신뢰도: {confidence:.3f}")

        if confidence > 0.1:
            print("✅ 차선이 검출되었습니다!")
        else:
            print("⚠️  차선 검출이 어렵습니다.")

    except Exception as e:
        print(f"❌ 에러: {e}")


if __name__ == "__main__":
    # 웹 인터페이스 실행
    interface = main()
    display(interface)